# Trabajo Práctico de Laboratorio N° 2

## Analizador Sintáctico Descendente Predictivo por Tabla

---
# 1. Verificación y modificación de la gramática

## 1.1. Gramática original

La gramática original de Mini-C proporcionada en el enunciado es:

```text
Program → DeclList

DeclList → DeclList Decl
         | Decl

Decl → FuncDecl
     | VarDecl

VarDecl → Type id ;
        | Type id = Expr ;

FuncDecl → Type id ( ParamList ) Block
         | Type id ( ) Block

Type → int | float | bool | void | string

ParamList → ParamList , Param
          | Param

Param → Type id

Block → { StmtList }

StmtList → StmtList Stmt
         | Stmt
         | λ

Stmt → VarDecl
     | id = Expr ;
     | id ( ArgList ) ;
     | id ( ) ;
     | IfStmt
     | WhileStmt
     | ForStmt
     | return Expr ;
     | return ;
     | print ( Expr ) ;
     | read ( id ) ;

IfStmt → if ( Expr ) Block else Block
       | if ( Expr ) Block

WhileStmt → while ( Expr ) Block

ForStmt → for ( id = Expr ; Expr ; id = Expr ) Block

Expr → Expr || RelExpr
     | Expr && RelExpr
     | RelExpr

RelExpr → AddExpr RelOp AddExpr
        | AddExpr

RelOp → == | != | < | > | <= | >=

AddExpr → AddExpr + Term
        | AddExpr - Term
        | Term

Term → Term * Unary
     | Term / Unary
     | Unary

Unary → ! Unary
      | - Unary
      | Factor

Factor → ( Expr )
        | id ( ArgList )
        | id ( )
        | id
        | num
        | str
        | true
        | false

ArgList → ArgList , Expr
        | Expr

### (El símbolo distinguido de la gramática es: Program)

## 1.2. Eliminación de recursividad izquierda

Se identificaron las siguientes producciones con recursividad izquierda directa:

* DeclList
* ParamList
* StmtList
* Expr
* AddExpr
* Term
* ArgList

Se aplicó el procedimiento de eliminación de recursividad izquierda, obteniendo:

-> DeclList
DeclList → Decl DeclList'
DeclList' → Decl DeclList'
          | λ

->ParamList
ParamList → Param ParamList'
ParamList' → , Param ParamList'
           | λ

->StmtList
StmtList → Stmt StmtList'
StmtList' → Stmt StmtList'
          | λ

->Expr
Expr → RelExpr Expr'
Expr' → || RelExpr Expr'
      | && RelExpr Expr'
      | λ

->AddExpr
AddExpr → Term AddExpr'
AddExpr' → + Term AddExpr'
         | - Term AddExpr'
         | λ

->Term
Term → Unary Term'
Term' → * Unary Term'
      | / Unary Term'
      | λ

->ArgList
ArgList → Expr ArgList'
ArgList' → , Expr ArgList'
         | λ

## 1.3. Factorizacion
Luego de eliminar la recursividad izquierda, se realizó la factorización de las producciones que presentan prefijos comunes.

->VarDecl
VarDecl → Type id A
A → ;
  | = Expr ;

B → ParamList ) Block
  | ) Block

->Decl
Decl → Type id C
C → ( B
  | A



->Stmt
Stmt → VarDecl
     | id I
     | IfStmt
     | WhileStmt
     | ForStmt
     | return J
     | print ( Expr ) ;
     | read ( id ) ;
I → = Expr ;
  | ( K
K → ArgList ) ;
  | ) ;
J → Expr ;
  | ;

->IfStmt
  IfStmt → if ( Expr ) Block D
D → else Block
  | λ

->RelExpr
RelExpr → AddExpr E
E → RelOp AddExpr
  | λ

->RelOp
RelOp → ==
      | !=
      | < F
      | > F
F → =
  | λ

->Factor
Factor → ( Expr )
       | id G
       | num
       | str
       | true
       | false
G → ( H
  | λ
H → ArgList )
  | )

## 1.4. Gramática modificada
Despues de eliminar la recursividad izquierda y realizar las factorizaciones necesarias, la gramática utilizada para el análisis sintáctico queda:

Program → DeclList

DeclList → Decl DeclList'

DeclList' → Decl DeclList'
          | λ

Decl → Type id C

C → ( B
  | A

A → ;
  | = Expr ;

VarDecl → Type id A



B → ParamList ) Block
  | ) Block

Type → int | float | bool | void | string

ParamList → Param ParamList'

ParamList' → , Param ParamList'
           | λ

Param → Type id

Block → { StmtList }

StmtList → Stmt StmtList'

StmtList' → Stmt StmtList'
          | λ

Stmt → VarDecl
     | id I
     | IfStmt
     | WhileStmt
     | ForStmt
     | return J
     | print ( Expr ) ;
     | read ( id ) ;

I → = Expr ;
  | ( K

K → ArgList ) ;
  | ) ;

J → Expr ;
  | ;

IfStmt → if ( Expr ) Block D

D → else Block
  | λ

WhileStmt → while ( Expr ) Block

ForStmt → for ( id = Expr ; Expr ; id = Expr ) Block

Expr → RelExpr Expr'

Expr' → || RelExpr Expr'
      | && RelExpr Expr'
      | λ

RelExpr → AddExpr E

E → RelOp AddExpr
  | λ

RelOp → ==
      | !=
      | < F
      | > F

F → =
  | λ

AddExpr → Term AddExpr'

AddExpr' → + Term AddExpr'
         | - Term AddExpr'
         | λ

Term → Unary Term'

Term' → * Unary Term'
      | / Unary Term'
      | λ

Unary → ! Unary
      | - Unary
      | Factor

Factor → ( Expr )
       | id G
       | num
       | str
       | true
       | false

G → ( H
  | λ

H → ArgList )
  | )

ArgList → Expr ArgList'

ArgList' → , Expr ArgList'
         | λ

# 2. primeros

## 2. Conjuntos Primeros

| No terminal | Primeros |
|---|---|
| `Program` | `{ int, float, bool, void, string }` |
| `DeclList` | `{ int, float, bool, void, string }` |
| `DeclList'` | `{ int, float, bool, void, string, λ }` |
| `Decl` | `{ int, float, bool, void, string }` |
| `C` | `{ (, ;, = }` |
| `A` | `{ ;, = }` |
| `VarDecl` | `{ int, float, bool, void, string }` |
| `B` | `{ int, float, bool, void, string, ) }` |
| `Type` | `{ int, float, bool, void, string }` |
| `ParamList` | `{ int, float, bool, void, string }` |
| `ParamList'` | `{ ,, λ }` |
| `Param` | `{ int, float, bool, void, string }` |
| `Block` | `{ { }` |
| `StmtList` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, λ }` |
| `StmtList'` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, λ }` |
| `Stmt` | `{ int, float, bool, void, string, id, if, while, for, return, print, read }` |
| `I` | `{ =, ( }` |
| `K` | `{ !, -, (, id, num, str, true, false, ) }` |
| `J` | `{ !, -, (, id, num, str, true, false, ; }` |
| `IfStmt` | `{ if }` |
| `D` | `{ else, λ }` |
| `WhileStmt` | `{ while }` |
| `ForStmt` | `{ for }` |
| `Expr` | `{ !, -, (, id, num, str, true, false }` |
| `Expr'` | `{ \|\|, &&, λ }` |
| `RelExpr` | `{ !, -, (, id, num, str, true, false }` |
| `E` | `{ ==, !=, <, >, λ }` |
| `RelOp` | `{ ==, !=, <, > }` |
| `F` | `{ =, λ }` |
| `AddExpr` | `{ !, -, (, id, num, str, true, false }` |
| `AddExpr'` | `{ +, -, λ }` |
| `Term` | `{ !, -, (, id, num, str, true, false }` |
| `Term'` | `{ *, /, λ }` |
| `Unary` | `{ !, -, (, id, num, str, true, false }` |
| `Factor` | `{ (, id, num, str, true, false }` |
| `G` | `{ (, λ }` |
| `H` | `{ !, -, (, id, num, str, true, false, ) }` |
| `ArgList` | `{ !, -, (, id, num, str, true, false }` |
| `ArgList'` | `{ ,, λ }` |

3. Siguientes
Para calcular los "Siguientes" usamos las siguientes reglas:

Regla 1
Si A es el símbolo inicial, entonces:
# ∈ FOLLOW(A)

Regla 2
Si existe una producción:
B → α A β
entonces:
Primeros(β) - {λ} ⊆ Siguientes(A)

Regla 3
Si existe una producción:
B → α A
o:
B → α A β
donde:
β ⇒* λ
entonces:
Siguientes(B) ⊆ Siguientes(A)



## 3. Conjuntos Siguientes

| No terminal | Siguientes |
|---|---|
| `Program` | `{ # }` |
| `DeclList` | `{ # }` |
| `DeclList'` | `{ # }` |
| `Decl` | `{ int, float, bool, void, string, # }` |
| `Type` | `{ id }` |
| `C` | `{ int, float, bool, void, string, # }` |
| `A` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, }, # }` |
| `B` | `{ int, float, bool, void, string, # }` |
| `VarDecl` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `ParamList` | `{ ) }` |
| `ParamList'` | `{ ) }` |
| `Param` | `{ ,, ) }` |
| `Block` | `{ else, int, float, bool, void, string, id, if, while, for, return, print, read, }, # }` |
| `StmtList` | `{ } }` |
| `StmtList'` | `{ } }` |
| `Stmt` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `I` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `K` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `J` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `IfStmt` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `D` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `WhileStmt` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `ForStmt` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `Expr` | `{ ), ,, ; }` |
| `Expr'` | `{ ), ,, ; }` |
| `RelExpr` | `{ ||, &&, ), ,, ; }` |
| `E` | `{ ||, &&, ), ,, ; }` |
| `RelOp` | `{ !, -, (, id, num, str, true, false }` |
| `F` | `{ !, -, (, id, num, str, true, false }` |
| `AddExpr` | `{ ==, !=, <, >, ||, &&, ), ,, ; }` |
| `AddExpr'` | `{ ==, !=, <, >, ||, &&, ), ,, ; }` |
| `Term` | `{ +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `Term'` | `{ +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `Unary` | `{ *, /, +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `Factor` | `{ *, /, +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `G` | `{ *, /, +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `H` | `{ *, /, +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `ArgList` | `{ ) }` |
| `ArgList'` | `{ ) }` |







## 2. Símbolos directrices

Los símbolos directrices permiten determinar qué producción debe seleccionar el analizador sintáctico predictivo cuando se encuentra un determinado no terminal y observa el próximo token de entrada.

Para una producción A → α se utilizan las siguientes reglas:

- Si λ no pertenece a FIRST(α), entonces:

  DS(A → α) = FIRST(α)

- Si λ pertenece a FIRST(α), entonces:

  DS(A → α) = (FIRST(α) - {λ}) ∪ FOLLOW(A)

En particular, para una producción A → λ:

  DS(A → λ) = FOLLOW(A)

Estos conjuntos se utilizarán posteriormente para construir la tabla de análisis sintáctico predictivo. Para que la gramática sea LL(1), las producciones alternativas de un mismo no terminal deben tener conjuntos de símbolos directrices disjuntos.

### 2.1. Cálculo de los símbolos directrices

Los símbolos directrices de las producciones de la gramática se calculan a partir de los conjuntos FIRST y FOLLOW obtenidos anteriormente.

| Producción | Símbolos directrices |
|---|---|
| Program → DeclList | { int, float, bool, void, string } |
| DeclList → Decl DeclList' | { int, float, bool, void, string } |
| DeclList' → Decl DeclList' | { int, float, bool, void, string } |
| DeclList' → λ | { # } |
| Decl → Type id C | { int, float, bool, void, string } |
| C → ( B | { ( } |
| C → A | { ;, = } |
| A → ; | { ; } |
| A → = Expr ; | { = } |
| Type → int | { int } |
| Type → float | { float } |
| Type → bool | { bool } |
| Type → void | { void } |
| Type → string | { string } |
| B → ParamList ) Block | { int, float, bool, void, string } |
| B → ) Block | { ) } |
| ParamList → Param ParamList' | { int, float, bool, void, string } |
| ParamList' → , Param ParamList' | { , } |
| ParamList' → λ | { ) } |
| Param → Type id | { int, float, bool, void, string } |
| Block → { StmtList } | { { } |
| StmtList → Stmt StmtList' | { int, float, bool, void, string, id, if, while, for, return, print, read } |
| StmtList → λ | { } } |
| StmtList' → Stmt StmtList' | { int, float, bool, void, string, id, if, while, for, return, print, read } |
| StmtList' → λ | { } } |
| Stmt → VarDecl | { int, float, bool, void, string } |
| Stmt → id I | { id } |
| Stmt → IfStmt | { if } |
| Stmt → WhileStmt | { while } |
| Stmt → ForStmt | { for } |
| Stmt → return J | { return } |
| Stmt → print ( Expr ) ; | { print } |
| Stmt → read ( id ) ; | { read } |
| I → = Expr ; | { = } |
| I → ( K | { ( } |
| K → ArgList ) ; | { !, -, (, id, num, str, true, false } |
| K → ) ; | { ) } |
| J → Expr ; | { !, -, (, id, num, str, true, false } |
| J → ; | { ; } |
| IfStmt → if ( Expr ) Block D | { if } |
| D → else Block | { else } |
| D → λ | { int, float, bool, void, string, id, if, while, for, return, print, read, } } |
| WhileStmt → while ( Expr ) Block | { while } |
| ForStmt → for ( id = Expr ; Expr ; id = Expr ) Block | { for } |
| Expr → RelExpr Expr' | { !, -, (, id, num, str, true, false } |
| Expr' → \|\| RelExpr Expr' | { \|\| } |
| Expr' → && RelExpr Expr' | { && } |
| Expr' → λ | { ), ,, ; } |
| RelExpr → AddExpr E | { !, -, (, id, num, str, true, false } |
| E → RelOp AddExpr | { ==, !=, <, > } |
| E → λ | { \|\|, &&, ), ,, ; } |
| RelOp → == | { == } |
| RelOp → != | { != } |
| RelOp → < F | { < } |
| RelOp → > F | { > } |
| F → = | { = } |
| F → λ | { !, -, (, id, num, str, true, false } |
| AddExpr → Term AddExpr' | { !, -, (, id, num, str, true, false } |
| AddExpr' → + Term AddExpr' | { + } |
| AddExpr' → - Term AddExpr' | { - } |
| AddExpr' → λ | { ==, !=, <, >, \|\|, &&, ), ,, ; } |
| Term → Unary Term' | { !, -, (, id, num, str, true, false } |
| Term' → * Unary Term' | { * } |
| Term' → / Unary Term' | { / } |
| Term' → λ | { +, -, ==, !=, <, >, \|\|, &&, ), ,, ; } |
| Unary → ! Unary | { ! } |
| Unary → - Unary | { - } |
| Unary → Factor | { (, id, num, str, true, false } |
| Factor → ( Expr ) | { ( } |
| Factor → id G | { id } |
| Factor → num | { num } |
| Factor → str | { str } |
| Factor → true | { true } |
| Factor → false | { false } |
| G → ( H | { ( } |
| G → λ | { *, /, +, -, ==, !=, <, >, \|\|, &&, ), ,, ; } |
| H → ArgList ) | { !, -, (, id, num, str, true, false } |
| H → ) | { ) } |
| ArgList → Expr ArgList' | { !, -, (, id, num, str, true, false } |
| ArgList' → , Expr ArgList' | { , } |
| ArgList' → λ | { ) } |

### 2.2. Verificación de conflictos LL(1)

Para que una gramática sea LL(1), las producciones alternativas de un mismo no terminal deben tener conjuntos de símbolos directrices disjuntos.

Al comparar los símbolos directrices de las producciones alternativas de cada no terminal, no se presentan intersecciones entre las alternativas.

Por lo tanto, la gramática modificada no presenta conflictos en los símbolos directrices y puede ser utilizada para construir una tabla de análisis sintáctico predictivo LL(1).



## 3. Tabla de análisis sintáctico predictivo

La tabla de análisis sintáctico predictivo permite determinar qué producción debe aplicar el analizador sintáctico a partir del no terminal que se encuentra en la cima de la pila y del próximo token de entrada.

La tabla se representa como M[A, a], donde:

- A es un no terminal de la gramática.
- a es un terminal o el símbolo de fin de entrada `#`.
- M[A, a] contiene la producción que debe utilizarse.

Para construir la tabla se utilizan los símbolos directrices calculados anteriormente. Para cada producción A → α, se coloca dicha producción en todas las columnas correspondientes a los símbolos pertenecientes a su conjunto de símbolos directrices.

Si una celda de la tabla queda vacía, significa que no existe una producción válida para esa combinación de no terminal y token de entrada.

En una gramática LL(1), cada celda de la tabla debe contener como máximo una producción. Si una misma celda recibe dos producciones diferentes, existe un conflicto y la gramática no es LL(1).

### 3.1. Construcción de la tabla

La tabla predictiva se construye utilizando los conjuntos de símbolos directrices obtenidos en la sección anterior.

Por ejemplo:

- Para `Program → DeclList`, los símbolos directrices son `{ int, float, bool, void, string }`. Por lo tanto, esa producción se coloca en las columnas correspondientes a esos cinco tokens.
- Para `DeclList' → λ`, el conjunto de símbolos directrices es `{ # }`, por lo que la producción se coloca en la columna `#`.
- Para `Expr' → λ`, los símbolos directrices son `{ ), ,, ; }`, por lo que la producción se coloca en cada una de esas columnas.

De esta manera se completa la tabla para todos los no terminales de la gramática.

### 3.2. Tabla predictiva

A partir de los símbolos directrices obtenidos anteriormente, se construyen las entradas de la tabla M[A, a].

En la siguiente tabla se muestran las celdas que contienen una producción. Los tokens que no aparecen para un determinado no terminal representan celdas de error.

| No terminal | Token de entrada | Producción |
|---|---|---|
| Program | int, float, bool, void, string | Program → DeclList |
| DeclList | int, float, bool, void, string | DeclList → Decl DeclList' |
| DeclList' | int, float, bool, void, string | DeclList' → Decl DeclList' |
| DeclList' | # | DeclList' → λ |
| Decl | int, float, bool, void, string | Decl → Type id C |
| C | ( | C → ( B |
| C | ;, = | C → A |
| A | ; | A → ; |
| A | = | A → = Expr ; |
| Type | int | Type → int |
| Type | float | Type → float |
| Type | bool | Type → bool |
| Type | void | Type → void |
| Type | string | Type → string |
| B | int, float, bool, void, string | B → ParamList ) Block |
| B | ) | B → ) Block |
| ParamList | int, float, bool, void, string | ParamList → Param ParamList' |
| ParamList' | , | ParamList' → , Param ParamList' |
| ParamList' | ) | ParamList' → λ |
| Param | int, float, bool, void, string | Param → Type id |
| Block | { | Block → { StmtList } |
| StmtList | int, float, bool, void, string, id, if, while, for, return, print, read | StmtList → Stmt StmtList' |
| StmtList | } | StmtList → λ |
| StmtList' | int, float, bool, void, string, id, if, while, for, return, print, read | StmtList' → Stmt StmtList' |
| StmtList' | } | StmtList' → λ |
| Stmt | int, float, bool, void, string | Stmt → VarDecl |
| Stmt | id | Stmt → id I |
| Stmt | if | Stmt → IfStmt |
| Stmt | while | Stmt → WhileStmt |
| Stmt | for | Stmt → ForStmt |
| Stmt | return | Stmt → return J |
| Stmt | print | Stmt → print ( Expr ) ; |
| Stmt | read | Stmt → read ( id ) ; |
| I | = | I → = Expr ; |
| I | ( | I → ( K |
| K | !, -, (, id, num, str, true, false | K → ArgList ) ; |
| K | ) | K → ) ; |
| J | !, -, (, id, num, str, true, false | J → Expr ; |
| J | ; | J → ; |
| IfStmt | if | IfStmt → if ( Expr ) Block D |
| D | else | D → else Block |
| D | int, float, bool, void, string, id, if, while, for, return, print, read, } | D → λ |
| WhileStmt | while | WhileStmt → while ( Expr ) Block |
| ForStmt | for | ForStmt → for ( id = Expr ; Expr ; id = Expr ) Block |
| Expr | !, -, (, id, num, str, true, false | Expr → RelExpr Expr' |
| Expr' | \|\| | Expr' → \|\| RelExpr Expr' |
| Expr' | && | Expr' → && RelExpr Expr' |
| Expr' | ), ,, ; | Expr' → λ |
| RelExpr | !, -, (, id, num, str, true, false | RelExpr → AddExpr E |
| E | ==, !=, <, > | E → RelOp AddExpr |
| E | \|\|, &&, ), ,, ; | E → λ |
| RelOp | == | RelOp → == |
| RelOp | != | RelOp → != |
| RelOp | < | RelOp → < F |
| RelOp | > | RelOp → > F |
| F | = | F → = |
| F | !, -, (, id, num, str, true, false | F → λ |
| AddExpr | !, -, (, id, num, str, true, false | AddExpr → Term AddExpr' |
| AddExpr' | + | AddExpr' → + Term AddExpr' |
| AddExpr' | - | AddExpr' → - Term AddExpr' |
| AddExpr' | ==, !=, <, >, \|\|, &&, ), ,, ; | AddExpr' → λ |
| Term | !, -, (, id, num, str, true, false | Term → Unary Term' |
| Term' | * | Term' → * Unary Term' |
| Term' | / | Term' → / Unary Term' |
| Term' | +, -, ==, !=, <, >, \|\|, &&, ), ,, ; | Term' → λ |
| Unary | ! | Unary → ! Unary |
| Unary | - | Unary → - Unary |
| Unary | (, id, num, str, true, false | Unary → Factor |
| Factor | ( | Factor → ( Expr ) |
| Factor | id | Factor → id G |
| Factor | num | Factor → num |
| Factor | str | Factor → str |
| Factor | true | Factor → true |
| Factor | false | Factor → false |
| G | ( | G → ( H |
| G | *, /, +, -, ==, !=, <, >, \|\|, &&, ), ,, ; | G → λ |
| H | !, -, (, id, num, str, true, false | H → ArgList ) |
| H | ) | H → ) |
| ArgList | !, -, (, id, num, str, true, false | ArgList → Expr ArgList' |
| ArgList' | , | ArgList' → , Expr ArgList' |
| ArgList' | ) | ArgList' → λ |

Las celdas no especificadas en la tabla representan situaciones de error sintáctico, ya que no existe una producción aplicable para esa combinación de no terminal y token de entrada.





## 4. Algoritmo de análisis sintáctico con tabla

El analizador sintáctico descendente predictivo por tabla utiliza una pila y la tabla de análisis sintáctico para determinar qué producción aplicar.

La cadena de entrada se completa agregando el símbolo de fin de entrada #. La pila se inicializa con dos símbolos: # en el fondo y el símbolo distinguido de la gramática, Program, en el tope.

En cada paso se considera el símbolo t apuntado en la cadena de entrada, el tope de la pila y la tabla M.

El algoritmo es el siguiente:

1. Si el tope de la pila es un terminal:
   - Si el tope coincide con t, se realiza Pop() y se avanza al siguiente símbolo de entrada.
   - Si no coincide, se informa un error.

2. Si el tope de la pila es un no terminal:
   - Se consulta la tabla M utilizando el símbolo de entrada y el no terminal del tope.
   - Si la tabla contiene una producción A → Y1 Y2 ... Yk, se realiza Pop() y luego se agregan a la pila los símbolos Yk, ..., Y2, Y1, de manera que Y1 quede en el tope.
   - Si no existe una producción correspondiente, se informa un error.

3. El análisis continúa hasta que tanto el tope de la pila como el símbolo de entrada sean #.

Este procedimiento permite realizar el análisis de forma descendente y predictiva, seleccionando una única producción a partir del no terminal en el tope de la pila y del siguiente símbolo de entrada.

### 4.1. Pseudocódigo del algoritmo

```text
t ← primer símbolo de la entrada
Pila ← [#, Program]

repeat
    if tope ∈ VT then
        if tope == t then
            Pop()
            t ← t + 1
        else
            return Error
        end if
    else
        if M(tope, t) == tope → Y1 Y2 ... Yk then
            Pop()
            Push(Yk, ..., Y2, Y1)
        else
            return Error
        end if
    end if
until tope == # ∧ t == #